# RelCheck — Visual Genome Recall Test

Tests whether RelCheck correctly **detects** naturally-occurring relational hallucinations
in BLIP-2 captions, using Visual Genome ground-truth relation annotations.

## How it works
1. Download VG relationships.json (~50 MB compressed)
2. Pick N_IMAGES images with >= MIN_VG_RELS diverse VG relations
3. Generate BLIP-2 captions for those images
4. Extract Llama triples from each caption
5. Match caption triples to VG ground truth — identify **contradictions** (ground truth hallucinations)
6. Run RelCheck VQA detection on each contradiction triple
7. Compute: **recall = hallucinations detected / total ground truth hallucinations**

## What this proves
Synthetic injection tests confirm recall on known fabrications. This confirms recall on **naturally-occurring** captioner hallucinations — a harder, more honest test.


In [ ]:
# ============================================================
# Cell 1 — Setup
# ============================================================
!pip install together Pillow requests -q

import os, json, re, time, random, requests, base64, zipfile, io
from pathlib import Path
from collections import Counter
from io import BytesIO
from PIL import Image
from together import Together
from google.colab import drive

# ── Config ──────────────────────────────────────────────────
TOGETHER_API_KEY = ''   # <-- paste your Together.ai key
N_IMAGES         = 20   # VG images to test
MIN_VG_RELS      = 3    # min VG relations per image (richer = better signal)
CAPTIONER        = 'qwen'   # 'qwen' | 'llava' — captioner to evaluate
SAVE_DIR         = '/content/drive/MyDrive/RelCheck_Data/vg_recall'
VG_DATA_PATH     = f'{SAVE_DIR}/relationships_sample.json'
LOCAL_IMAGES_ZIP = f'{SAVE_DIR}/images2.zip'   # your downloaded VG images zip

LLM_MODEL    = 'meta-llama/Llama-3.3-70B-Instruct-Turbo'
VLM_MODEL    = 'Qwen/Qwen3-VL-8B-Instruct'          # used for VQA verification
# Captioner models (Together.ai VLM API — no local GPU load needed)
_CAPTIONER_MODELS = {
    'qwen':  'Qwen/Qwen3-VL-8B-Instruct',
    'llava': 'llava-hf/llava-1.5-7b-hf',
}
CAPTION_MODEL = _CAPTIONER_MODELS[CAPTIONER]

drive.mount('/content/drive')
os.makedirs(SAVE_DIR, exist_ok=True)
os.environ['TOGETHER_API_KEY'] = TOGETHER_API_KEY
client = Together(api_key=TOGETHER_API_KEY)

def llm_call(messages, model=LLM_MODEL, max_tokens=600, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=0.0, max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f'  API failed: {e}')
                return None

def vlm_call(messages, max_tokens=10, retries=3):
    return llm_call(messages, model=VLM_MODEL, max_tokens=max_tokens, retries=retries)

print('Setup done.')


In [ ]:
# ============================================================
# Cell 2 — Load VG relationships + select candidate images
# ============================================================
# Strategy:
#   1. Scan LOCAL_IMAGES_ZIP first → get available image IDs
#   2. Load all of relationships.json, keep only entries whose
#      image_id is in the available set  (no STREAM_LIMIT needed)
#   3. Score and rank those candidates

# VG data moved from Stanford to UW (Ranjay Krishna's server).
VG_REL_URLS = [
    'https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/relationships_v1_2.json.zip',
    'https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/relationships_v1.json.zip',
]
# Path to manually downloaded relationships zip on Drive (or None to auto-download)
LOCAL_ZIP_PATH = f'{SAVE_DIR}/relationships.json.zip'

SKIP_PREDS = {'has','have','of','is','are','was','were','with',
              'and','a','an','the','it','its','for','to','by',
              # Too vague — nearly always COMPATIBLE with any caption description
              'near','nearby','beside','close to','adjacent to',
              'around','between','among',
              'part of','made of','attached to','connected to',
              'belonging to','painted on','written on',
              'along','across','through','toward'}

# Relations specific enough to actually contradict a captioner.
# We score images higher for having more of these — they produce
# real, detectable hallucinations aligned with what RelCheck targets.
PREFERRED_PREDS = {
    # Directional spatial
    'left of','right of','above','below','under','on top of',
    'on','inside','outside','over','beneath','leaning on',
    # Action / interaction (clearly verifiable by VQA)
    'riding','holding','carrying','wearing','eating','drinking',
    'sitting on','standing on','lying on','walking on','jumping over',
    'kicking','touching','pulling','pushing','hugging','kissing',
    'feeding','reading','watching','playing with','using',
    'mounted on','chained to',
}

def clean_vg_name(name):
    name = name.lower().strip()
    for art in ['a ', 'an ', 'the ']:
        if name.startswith(art):
            name = name[len(art):]
    return name.strip()

# ── Step 1: scan the images zip to know which IDs we actually have ─────
zip_available_ids = set()
if LOCAL_IMAGES_ZIP and Path(LOCAL_IMAGES_ZIP).exists():
    print(f'Scanning {LOCAL_IMAGES_ZIP} for available image IDs...')
    with zipfile.ZipFile(LOCAL_IMAGES_ZIP, 'r') as zf:
        for name in zf.namelist():
            stem = Path(name).stem
            try:
                zip_available_ids.add(int(stem))
            except ValueError:
                pass
    print(f'  Found {len(zip_available_ids)} images in zip '
          f'(IDs {min(zip_available_ids)}–{max(zip_available_ids)})')
else:
    print('WARNING: LOCAL_IMAGES_ZIP not found. '
          'Set LOCAL_IMAGES_ZIP in Cell 1 to your images2.zip path.')

# ── Step 2: load relationships.json, filtered to available IDs ─────────

if Path(VG_DATA_PATH).exists():
    with open(VG_DATA_PATH) as f:
        vg_data = json.load(f)
    print(f'Loaded {len(vg_data)} VG entries from cache')
    # Warn if cache was built without zip filtering
    cached_ids = {e['image_id'] for e in vg_data}
    overlap = len(cached_ids & zip_available_ids) if zip_available_ids else len(cached_ids)
    if zip_available_ids and overlap == 0:
        print('  WARNING: cached data has no overlap with your zip — deleting cache.')
        Path(VG_DATA_PATH).unlink()
        vg_data = None
    else:
        print(f'  {overlap}/{len(cached_ids)} cached entries overlap with zip')
else:
    vg_data = None

if vg_data is None:
    buf = None
    if LOCAL_ZIP_PATH and Path(LOCAL_ZIP_PATH).exists():
        print(f'Found local relationships zip at {LOCAL_ZIP_PATH}')
        with open(LOCAL_ZIP_PATH, 'rb') as zf:
            buf = io.BytesIO(zf.read())
    else:
        print('Downloading VG relationships (trying multiple URLs)...')
        for url in VG_REL_URLS:
            try:
                print(f'  Trying {url}')
                resp = requests.get(url, stream=True, timeout=60)
                resp.raise_for_status()
                buf = io.BytesIO()
                total = 0
                for chunk in resp.iter_content(chunk_size=1024*1024):
                    buf.write(chunk)
                    total += len(chunk)
                    print(f'  {total/1e6:.1f} MB...', end='\r')
                print(f'  Downloaded {total/1e6:.1f} MB')
                break
            except Exception as e:
                print(f'  Failed: {e}')
                buf = None

    if buf is None:
        raise RuntimeError(
            'Could not load relationships.json.\n'
            'Please download from https://visualgenome.org/api/v0/api_home\n'
            f'and upload to Drive as: {LOCAL_ZIP_PATH}'
        )

    buf.seek(0)
    print('Parsing relationships.json...')
    with zipfile.ZipFile(buf) as zf:
        fname = next(n for n in zf.namelist() if n.endswith('.json'))
        raw = json.loads(zf.open(fname).read().decode('utf-8'))

    if zip_available_ids:
        vg_data = [e for e in raw if e['image_id'] in zip_available_ids]
        print(f'Filtered {len(raw)} total → {len(vg_data)} entries matching your zip')
    else:
        vg_data = raw[:5000]   # fallback if no zip scanned
        print(f'No zip scanned — using first {len(vg_data)} entries as fallback')

    with open(VG_DATA_PATH, 'w') as f:
        json.dump(vg_data, f)
    print(f'Saved {len(vg_data)} entries to cache')

# ── Step 3: score and rank candidates ─────────────────────────────────
candidates = []
for entry in vg_data:
    img_id = entry['image_id']
    rels   = entry.get('relationships', [])
    good   = [r for r in rels
              if r.get('predicate','').strip().lower() not in SKIP_PREDS
              and len(r.get('predicate','').split()) <= 4
              and r.get('subject',{}).get('name')
              and r.get('object',{}).get('name')]
    if len(good) < MIN_VG_RELS:
        continue
    # Score = preferred relation count (primary) + total distinct preds (tiebreak).
    # Preferred = specific spatial/action that captioners mention and can get wrong.
    # Vague relations like 'near', 'next to' are excluded (nearly always COMPATIBLE).
    pref_count = sum(1 for r in good
                     if r['predicate'].lower().strip() in PREFERRED_PREDS)
    total_preds = len({r['predicate'].lower() for r in good})
    score = pref_count * 100 + total_preds
    # Store only preferred + non-trivial relations as ground truth
    gt_rels = [r for r in good
               if r['predicate'].lower().strip() in PREFERRED_PREDS
               or r['predicate'].lower().strip() not in SKIP_PREDS]
    candidates.append((score, img_id, gt_rels[:10]))

candidates.sort(reverse=True)
candidates_by_id = {img_id: rels for _, img_id, rels in candidates}

print(f'\nTotal candidate images (>= {MIN_VG_RELS} good relations): {len(candidates_by_id)}')
if len(candidates_by_id) == 0:
    print('ERROR: 0 candidates found.')
    print(f'  zip_available_ids count: {len(zip_available_ids)}')
    print(f'  vg_data count: {len(vg_data)}')
    print('  Check that LOCAL_IMAGES_ZIP is the right zip (images2.zip = Part 2 = IDs 108250+).')
else:
    sample_ids = list(candidates_by_id.keys())[:5]
    print(f'  Sample IDs: {sample_ids}')
    for sc, iid, _ in candidates[:5]:
        pref = sc // 100
        print(f'    ID {iid}: {pref} preferred spatial/action rels (score={sc})')
    print(f'  Will select {min(N_IMAGES, len(candidates_by_id))} of these in Cell 3')

# ── Download image_data.json for Flickr URL fallback ──────────────────
IMAGE_DATA_PATH = f'{SAVE_DIR}/image_data.json'
IMAGE_DATA_URLS = [
    'https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/image_data.json.zip',
]

if Path(IMAGE_DATA_PATH).exists():
    with open(IMAGE_DATA_PATH) as f:
        image_data = json.load(f)
    print(f'\nLoaded image_data.json from cache ({len(image_data)} entries)')
else:
    print('\nDownloading image_data.json...')
    img_buf = None
    for url in IMAGE_DATA_URLS:
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
            img_buf = io.BytesIO(resp.content)
            print(f'  Downloaded {len(resp.content)/1e6:.1f} MB')
            break
        except Exception as e:
            print(f'  Failed: {e}')
    if img_buf is None:
        print('  Could not download image_data.json — Flickr fallback disabled')
        image_data = []
    else:
        img_buf.seek(0)
        with zipfile.ZipFile(img_buf) as zf:
            fname = next(n for n in zf.namelist() if n.endswith('.json'))
            image_data = json.loads(zf.open(fname).read().decode('utf-8'))
        with open(IMAGE_DATA_PATH, 'w') as f:
            json.dump(image_data, f)
        print(f'  Saved {len(image_data)} entries')

flickr_urls = {}
for entry in image_data:
    img_id = entry.get('image_id') or entry.get('id')
    url = entry.get('flickr_300k_url') or entry.get('url', '')
    if img_id and url:
        flickr_urls[img_id] = url
print(f'Flickr URLs mapped: {len(flickr_urls)} images')


In [ ]:
# ============================================================
# Cell 3 — Download VG images + caption via Together.ai VLM
# ============================================================
# Captions are generated via Together.ai API (no local GPU needed).
# Uses a detailed describe prompt so we get rich relational captions
# — much more likely to surface detectable hallucinations than
# BLIP-2's terse 10-word outputs.

CAPTIONS_PATH = f'{SAVE_DIR}/{CAPTIONER}_captions.json'

DESCRIBE_PROMPT = (
    'Describe this image in detail. Include all objects, '
    'their spatial positions relative to each other, any actions '
    'or interactions taking place, and notable attributes like colors and sizes.'
)

def caption_image(pil_image, model=CAPTION_MODEL, max_tokens=300):
    '''Generate a detailed caption via Together.ai VLM API.'''
    buf = BytesIO()
    pil_image.convert('RGB').save(buf, format='JPEG', quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return llm_call(
        [{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
            {'type': 'text', 'text': DESCRIBE_PROMPT},
        ]}],
        model=model, max_tokens=max_tokens,
    )

# ── Step 1: find available image IDs from zip or Flickr ─────────────
available_ids = set()
vg_zip = None   # will hold open ZipFile handle if using local zip

if LOCAL_IMAGES_ZIP and Path(LOCAL_IMAGES_ZIP).exists():
    print(f'Scanning zip: {LOCAL_IMAGES_ZIP} ...')
    vg_zip = zipfile.ZipFile(LOCAL_IMAGES_ZIP, 'r')
    for name in vg_zip.namelist():
        stem = Path(name).stem
        try:
            available_ids.add(int(stem))
        except ValueError:
            pass
    print(f'Found {len(available_ids)} images in zip')
else:
    available_ids = set(flickr_urls.keys())
    print(f'Using Flickr URLs: {len(available_ids)} images available')

# ── Step 2: select N_IMAGES from candidates that are available ───────
pool = [(score, img_id, rels) for score, img_id, rels in candidates
        if img_id in available_ids]
if len(pool) == 0:
    raise RuntimeError(
        'No candidates overlap with available images.\n'
        'Check that LOCAL_IMAGES_ZIP points to your VG_100K.zip or VG_100K_2.zip.'
    )
random.seed(42)
selected = random.sample(pool, min(N_IMAGES, len(pool)))
selected_images = {img_id: rels for _, img_id, rels in selected}
print(f'Selected {len(selected_images)}/{len(pool)} available candidates')

# ── Step 3: load selected images ─────────────────────────────────────
pil_images = {}
if vg_zip is not None:
    # Build filename→zippath lookup (handles subfolders inside zip)
    zip_index = {Path(n).stem: n for n in vg_zip.namelist() if n.endswith('.jpg')}
    for img_id in selected_images:
        zpath = zip_index.get(str(img_id))
        if zpath:
            try:
                pil_images[img_id] = Image.open(BytesIO(vg_zip.read(zpath))).convert('RGB')
            except Exception as e:
                print(f'  [{img_id}] load error: {e}')
        else:
            print(f'  [{img_id}] not found in zip')
else:
    print('Downloading selected images via Flickr URLs...')
    for img_id in selected_images:
        try:
            r = requests.get(flickr_urls[img_id], timeout=15)
            r.raise_for_status()
            pil_images[img_id] = Image.open(BytesIO(r.content)).convert('RGB')
        except Exception as e:
            print(f'  [{img_id}] failed: {e}')

print(f'Loaded {len(pil_images)}/{len(selected_images)} images')

# Generate captions
if Path(CAPTIONS_PATH).exists():
    with open(CAPTIONS_PATH) as f:
        captions = {int(k): v for k, v in json.load(f).items()}
    print(f'Loaded {len(captions)} {CAPTIONER} captions from cache')
else:
    print(f'Generating captions with {CAPTIONER} ({CAPTION_MODEL})...')
    captions = {}
    for img_id, pil in pil_images.items():
        cap = caption_image(pil)
        if cap:
            captions[img_id] = cap
            words = len(cap.split())
            print(f'  [{img_id}] {words}w: {cap[:80]}...')
        else:
            print(f'  [{img_id}] captioning failed')
    with open(CAPTIONS_PATH, 'w') as f:
        json.dump({str(k): v for k, v in captions.items()}, f)
    print(f'Captioned {len(captions)} images')

avg_words = sum(len(c.split()) for c in captions.values()) / max(len(captions), 1)
print(f'Avg caption length: {avg_words:.0f} words ({CAPTIONER})')
if captions:
    print('Example caption:')
    ex_id = list(captions.keys())[0]
    print(f'  [{ex_id}] {captions[ex_id][:200]}')
else:
    print('WARNING: No captions generated — check TOGETHER_API_KEY and CAPTIONER model ID')


In [ ]:
# ============================================================
# Cell 4 — Extract caption triples + identify GT hallucinations
# ============================================================
# A caption triple is a GT hallucination if:
#   - Same (subject, object) pair exists in VG (entity match)
#   - But the relation in the caption CONTRADICTS the VG relation
# We use Llama to judge compatibility — handles paraphrases cleanly.

TRIPLE_PROMPT = '''Extract relational claims from this caption as JSON.
Caption: "{caption}"
Output a JSON array: [{{"subject": "...", "relation": "...", "object": "..."}}]
Only claims about how two entities relate. ONLY valid JSON, no explanation.'''

COMPAT_PROMPT = '''Two descriptions of the same entity pair in an image:
  Caption says: {s} {cap_rel} {o}
  Ground truth: {s} {vg_rel} {o}
Are these COMPATIBLE (both can be true simultaneously) or CONTRADICTORY (mutually exclusive)?

Rules:
- COMPATIBLE if they describe the same physical reality from different perspectives or levels of detail
- COMPATIBLE if one relation implies or is consistent with the other
- CONTRADICTORY only if the two relations CANNOT both be true at the same time

Examples:
  COMPATIBLE: 'man riding horse' + 'man on horse'  (same thing, different wording)
  COMPATIBLE: 'candle inserted into cake' + 'candle on top of cake'  (both true of a candle stuck in cake)
  COMPATIBLE: 'person sitting on chair' + 'person above chair'  (sitting implies above)
  COMPATIBLE: 'dog next to boy' + 'dog beside boy'  (synonyms)
  CONTRADICTORY: 'man left of table' + 'man right of table'  (mutually exclusive positions)
  CONTRADICTORY: 'cat on sofa' + 'cat under sofa'  (cannot be both simultaneously)
  CONTRADICTORY: 'person riding horse' + 'person standing next to horse'

Answer ONLY: COMPATIBLE or CONTRADICTORY'''

def entity_overlap(a, b):
    STOP = {'a','an','the','of','in','on','at','is','its','some','with'}
    wa = {w for w in a.lower().split() if w not in STOP and len(w) >= 3}
    wb = {w for w in b.lower().split() if w not in STOP and len(w) >= 3}
    if not wa or not wb:
        return a.lower().strip() == b.lower().strip()
    for w in wa:
        for v in wb:
            if w == v or (len(w) >= 4 and len(v) >= 4 and w[:4] == v[:4]):
                return True
    return False

GT_PATH     = f'{SAVE_DIR}/gt_hallucinations_{CAPTIONER}.json'
gt_hallucs  = {}   # img_id -> [{subject, cap_rel, object, vg_rel, claim}]
cap_triples = {}   # img_id -> [extracted triples]

# Set True to force re-run (e.g. after updating COMPAT_PROMPT)
FORCE_RERUN_GT = False
if not FORCE_RERUN_GT and Path(GT_PATH).exists():
    with open(GT_PATH) as f:
        saved = json.load(f)
    gt_hallucs  = {int(k): v for k, v in saved['hallucinations'].items()}
    cap_triples = {int(k): v for k, v in saved['cap_triples'].items()}
    print(f'Loaded from cache: {sum(len(v) for v in gt_hallucs.values())} GT hallucinations')
else:
    print('Extracting triples and matching against VG ground truth...')
    for img_id, caption in captions.items():
        vg_rels = selected_images.get(img_id, [])
        raw = llm_call([{'role': 'user', 'content': TRIPLE_PROMPT.format(caption=caption)}],
                       max_tokens=400)
        try:
            clean = re.sub(r'```[a-z]*\n?', '', raw or '').replace('```', '').strip()
            triples = json.loads(clean)
        except Exception:
            triples = []
        cap_triples[img_id] = triples

        hallucinations = []
        for ct in triples:
            cs = clean_vg_name(ct.get('subject', ''))
            cr = ct.get('relation', '').lower().strip()
            co = clean_vg_name(ct.get('object', ''))
            if not cs or not cr or not co:
                continue
            already_halluc = False   # don't double-count same caption triple
            for vr in vg_rels:
                if already_halluc:
                    break
                vs = clean_vg_name(vr['subject']['name'])
                vp = vr['predicate'].lower().strip()
                vo = clean_vg_name(vr['object']['name'])
                if not (entity_overlap(cs, vs) and entity_overlap(co, vo)):
                    continue
                # Same entity pair found in VG
                if cr == vp:
                    # Identical relation — definitely compatible; no need to check
                    # other VG relations for this pair (one clear match is enough).
                    already_halluc = False
                    break
                compat = (llm_call([{'role': 'user', 'content':
                    COMPAT_PROMPT.format(s=cs, cap_rel=cr, o=co, vg_rel=vp)}],
                    max_tokens=5) or '').upper()
                if 'COMPATIBLE' in compat:
                    # One compatible VG match → caption triple is not a hallucination.
                    already_halluc = False
                    break
                if 'CONTRADICTORY' in compat:
                    hallucinations.append({
                        'subject': cs, 'cap_rel': cr, 'object': co,
                        'vg_rel': vp, 'claim': f'{cs} {cr} {co}',
                    })
                    print(f'  [{img_id}] HALLUCINATION: ({cs}, {cr}, {co}) VG says "{vp}"')
                    already_halluc = True   # found one contradiction — done with this triple
        gt_hallucs[img_id] = hallucinations

    with open(GT_PATH, 'w') as f:
        json.dump({'hallucinations': {str(k): v for k, v in gt_hallucs.items()}, 'captioner': CAPTIONER,
                   'cap_triples':    {str(k): v for k, v in cap_triples.items()}}, f)

total_hall = sum(len(v) for v in gt_hallucs.values())
n_affected = sum(1 for v in gt_hallucs.values() if v)
print(f'\nGT hallucinations: {total_hall} across {n_affected}/{len(captions)} images')
if total_hall == 0:
    print('WARNING: 0 hallucinations found. Try N_IMAGES=50 or lowering MIN_VG_RELS.')


In [ ]:
# ============================================================
# Cell 5 — RelCheck VQA detection + recall
# ============================================================
# For each GT hallucination, run RelCheck's VQA verifier.
# We test detection only (no correction) — just: does RelCheck flag it INCORRECT?

RECALL_PATH = f'{SAVE_DIR}/recall_results_{CAPTIONER}.json'

def _img_b64(pil_image):
    buf = BytesIO()
    pil_image.convert('RGB').save(buf, format='JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode()

def check_triple_vqa(subj, rel, obj_, pil_image, n_questions=3):
    '''
    Run YES/NO VQA for (subj, rel, obj) against the image.
    Returns (detected: bool, yes_v, no_v, total)
    detected=True if majority vote says NO (relation is INCORRECT).
    '''
    if pil_image is None:
        return False, 0, 0, 0
    b64 = _img_b64(pil_image)
    questions = [
        f'Is the {subj} {rel} the {obj_}? Answer YES or NO only.',
        f'Does the {subj} appear to be {rel} the {obj_}? YES or NO.',
        f'In this image, is there a {subj} that is {rel} a {obj_}? YES or NO.',
    ][:n_questions]
    yes_v = no_v = 0
    for q in questions:
        r = vlm_call([{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
            {'type': 'text', 'text': q},
        ]}], max_tokens=5)
        if not r: continue
        rl = r.strip().lower()
        if 'yes' in rl and 'no' not in rl:
            yes_v += 1
        elif 'no' in rl:
            no_v += 1
    total = yes_v + no_v
    if total == 0:
        return False, 0, 0, 0
    yes_ratio = yes_v / total
    detected = yes_ratio < 0.50   # majority NO = flagged as INCORRECT
    return detected, yes_v, no_v, total

recall_results = []
total_hall = sum(len(v) for v in gt_hallucs.values())   # recompute in case Cell 4 was cached
print(f'Running VQA detection on {total_hall} GT hallucinations...\n')

for img_id, hallucs in gt_hallucs.items():
    if not hallucs:
        continue
    pil     = pil_images.get(img_id)
    caption = captions.get(img_id, '')
    for hall in hallucs:
        s, r, o = hall['subject'], hall['cap_rel'], hall['object']
        detected, yes_v, no_v, total = check_triple_vqa(s, r, o, pil)
        result = {
            'img_id': img_id, 'caption': caption,
            'subject': s, 'cap_rel': r, 'object': o,
            'vg_rel': hall['vg_rel'],
            'detected': detected,
            'votes': f'{yes_v}Y/{no_v}N/{total}T',
        }
        recall_results.append(result)
        status = 'DETECTED' if detected else 'MISSED   '
        print(f'  [{img_id}] {status} ({s}, {r}, {o}) VG="{hall["vg_rel"]}" [{yes_v}Y/{no_v}N]')

with open(RECALL_PATH, 'w') as f:
    json.dump(recall_results, f, indent=2)

# ── Text summary ─────────────────────────────────────────────
n_total    = len(recall_results)
n_detected = sum(1 for r in recall_results if r['detected'])
recall     = n_detected / max(n_total, 1)

print('\n' + '='*60)
print('RELCHECK RECALL — VISUAL GENOME GROUND TRUTH')
print('='*60)
print(f'  GT hallucinations : {n_total}')
print(f'  Detected          : {n_detected}')
print(f'  Recall            : {recall:.1%}')
print()
print('MISSED cases:')
missed = [r for r in recall_results if not r['detected']]
if not missed:
    print('  None — perfect recall!')
else:
    for r in missed:
        print(f'  [{r["img_id"]}] ({r["subject"]}, {r["cap_rel"]}, {r["object"]}) '
              f'VG="{r["vg_rel"]}" [{r["votes"]}]')
print('='*60)

# ── Visual results: image + caption + detection labels ────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

seen = set()
ordered_ids = [r['img_id'] for r in recall_results
               if not (r['img_id'] in seen or seen.add(r['img_id']))]

for img_id in ordered_ids:
    pil      = pil_images.get(img_id)
    caption  = captions.get(img_id, '(no caption)')
    img_results = [r for r in recall_results if r['img_id'] == img_id]
    if pil is None:
        continue

    fig, (ax_img, ax_txt) = plt.subplots(1, 2, figsize=(12, 4),
                                          gridspec_kw={'width_ratios': [1, 1]})

    # Left: image
    ax_img.imshow(pil)
    ax_img.axis('off')
    ax_img.set_title(f'Image {img_id}', fontsize=9)

    # Right: caption + results table
    ax_txt.axis('off')
    lines = []
    # Wrap caption ~10 words per line
    words = caption.split()
    for j in range(0, len(words), 10):
        lines.append(' '.join(words[j:j+10]))
    lines.append('')
    for r in img_results:
        color = 'green' if r['detected'] else 'red'
        icon  = '✓ DETECTED' if r['detected'] else '✗ MISSED'
        lines.append(f'{icon}')
        lines.append(f'  caption: {r["subject"]} {r["cap_rel"]} {r["object"]}')
        lines.append(f'  VG GT:   {r["subject"]} {r["vg_rel"]} {r["object"]}')
        lines.append(f'  votes:   {r["votes"]}')
        lines.append('')

    ax_txt.text(0.02, 0.98, '\n'.join(lines),
                transform=ax_txt.transAxes,
                fontsize=8.5, verticalalignment='top', fontfamily='monospace',
                wrap=True)

    plt.tight_layout()
    plt.show()
    plt.close()
